# Week 60, Day 1: k-means: clustering as minimising distortion

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w060/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
t = np.linspace(0, 2, 40)
shapes = np.array([
    np.exp(-((t - 0.3) / 0.12) ** 2),                   # transient
    1 / (1 + np.exp(-(t - 0.3) / 0.05)),                 # sustained
    np.clip(t / 2, 0, 1),                                # ramping
    -0.6 / (1 + np.exp(-(t - 0.3) / 0.05)),              # suppressed
])

def psths(n, g, noise=0.25):
    cls = g.integers(0, 4, n)
    gain = g.uniform(0.7, 1.3, n)
    return gain[:, None] * shapes[cls] + noise * g.normal(size=(n, 40)), cls

def match_accuracy(labels, truth, k):
    """Fraction agreeing with the truth under the best one-to-one renaming of clusters (greedy)."""
    C = np.array([[np.sum((labels == a) & (truth == b)) for b in range(k)] for a in range(k)])
    total = 0
    for _ in range(k):
        a, b = np.unravel_index(np.argmax(C), C.shape); total += C[a, b]; C[a, :] = -1; C[:, b] = -1
    return total / len(truth)

g0 = np.random.default_rng(601)
P, cls = psths(300, g0)
fig, ax = plt.subplots(1, 4, figsize=(9, 2), sharey=True)
for c in range(4):
    ax[c].plot(t, P[cls == c][:8].T, lw=0.7, alpha=0.6); ax[c].set_title(["transient", "sustained", "ramping", "suppressed"][c], fontsize=9)
plt.tight_layout(); plt.show()

### Exercise: Lloyd's algorithm

Write kmeans(X, k, g, iters=50): start from k randomly chosen points, then alternate the assignment and update steps, recording the distortion J after each assignment step. Return labels, centres and the history of J. Run it from 10 starts, keep the run with the lowest final J, and compare its clusters with the true classes.

In [ ]:
def kmeans(X, k, g, iters=50, init=None):
    C = X[g.choice(len(X), k, replace=False)] if init is None else init.copy()
    hist = []
    for _ in range(iters):
        d = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1)
        lab = ...
        hist.append(d[np.arange(len(X)), lab].sum())
        for j in range(k):
            if np.any(lab == j):
                C[j] = ...
        if len(hist) > 1 and hist[-1] == hist[-2]:
            break
    return lab, C, hist

gk = np.random.default_rng(6011)
runs = [kmeans(P, 4, gk) for _ in range(10)]
finals = [h[-1] for _, _, h in runs]
lab_km, C_km, history = runs[int(np.argmin(finals))]
acc_km = match_accuracy(lab_km, cls, 4)
print("final distortion of the 10 starts:", np.round(sorted(finals), 1))
print(f"best run: {len(history)} iterations, accuracy against the true classes {acc_km:.3f}")

In [ ]:
import numpy as np
def test_monotone():
    "the distortion never increases across iterations"
    assert all(b <= a + 1e-9 for a, b in zip(history, history[1:]))
def test_recovers():
    "the best of 10 starts recovers the four classes with accuracy above 0.9"
    assert acc_km > 0.9

test_monotone()
test_recovers()
print("All checks passed")

### Exercise: k-means++ seeding

Write kpp_init(X, k, g). Then, on data with eight well-separated groups, run 40 single starts of k-means from random points and 40 from k-means++ seeds, and count how often each reaches a distortion within 5% of the best found.

In [ ]:
def kpp_init(X, k, g):
    C = [X[g.integers(len(X))]]
    for _ in range(1, k):
        d2 = np.min(((X[:, None, :] - np.array(C)[None]) ** 2).sum(-1), axis=1)
        C.append(...)
    return np.array(C)

gs = np.random.default_rng(6012)
centres8 = 4 * gs.normal(size=(8, 2))
X8 = np.repeat(centres8, 40, axis=0) + 0.6 * gs.normal(size=(320, 2))
fin_rand = [kmeans(X8, 8, gs)[2][-1] for _ in range(40)]
fin_pp = [kmeans(X8, 8, gs, init=kpp_init(X8, 8, gs))[2][-1] for _ in range(40)]
good = 1.05 * min(fin_rand + fin_pp)
print(f"reached the good solution: random starts {np.mean(np.array(fin_rand) < good):.2f}, k-means++ {np.mean(np.array(fin_pp) < good):.2f}")

In [ ]:
import numpy as np
def test_seed():
    "kpp_init returns k distinct rows of X"
    C = kpp_init(P, 4, np.random.default_rng(0))
    assert C.shape == (4, 40) and len({tuple(np.round(c, 8)) for c in C}) == 4
def test_better():
    "with eight groups, k-means++ reaches the good solution more often than random starts"
    assert np.mean(np.array(fin_pp) < good) > np.mean(np.array(fin_rand) < good)

test_seed()
test_better()
print("All checks passed")

### Exercise: Unequal groups

Simulate a tight group of 200 neurons (sd 0.5) centred at 0 and a wide group of 40 (sd 2.5) centred at 4, in two dimensions. Run k-means with k = 2 from k-means++ seeds (best of 5) and measure the agreement with the true groups and how many wide-group points are captured by the tight cluster.

In [ ]:
gu = np.random.default_rng(6013)
Xu = np.vstack([0.5 * gu.normal(size=(200, 2)), np.array([4.0, 0]) + 2.5 * gu.normal(size=(40, 2))])
yu = np.r_[np.zeros(200, int), np.ones(40, int)]
runs_u = [kmeans(Xu, 2, gu, init=kpp_init(Xu, 2, gu)) for _ in range(5)]
lab_u, C_u, _ = min(runs_u, key=lambda run: run[2][-1])
acc_unequal = match_accuracy(lab_u, yu, 2)
tight_label = np.bincount(lab_u[:200]).argmax()
captured = ...
print(f"agreement {acc_unequal:.3f}; wide-group points assigned to the tight cluster: {captured} of 40")
print("centres:", np.round(C_u, 2))

In [ ]:
import numpy as np
def test_captured():
    "overall agreement looks high, yet k-means puts more than 15% of the wide group in the tight cluster"
    assert acc_unequal > 0.9 and captured > 6

test_captured()
print("All checks passed")